# CUDA fundamentals — coding quiz

Work from first principles. Attempt each section before reviewing pmpp.ipynb. Checks provide feedback, not solutions.

Focus: tensor layout, loops, row-major indexing, broadcasting, launch geometry, and the Python → C++/CUDA boundary.

In [34]:
import math
import torch

torch.manual_seed(7)

## 1. Image layout — fill the blanks

An RGB image has shape (3, h, w): channels first. After flattening contiguous storage, all red pixels appear first, then green, then blue. Complete the one-pixel grayscale calculation.

In [35]:
def rgb2grey_one_pixel(x, i):
    # x has shape (3, h, w); i is a flat spatial-pixel index
    _, h, w = x.shape
    n = h*w
    flat = x.flatten()
    # print(flat.shape)
    return 0.2989 * flat[i] + 0.5870 * flat[i+n] + 0.1140 * flat[i+2*n]

tiny = torch.tensor([[[10, 20], [30, 40]],
                     [[50, 60], [70, 80]],
                     [[90, 100], [110, 120]]], dtype=torch.float32)
expected_pixel_2 = torch.tensor(0.2989*30 + 0.5870*70 + 0.1140*110)
assert torch.isclose(rgb2grey_one_pixel(tiny, 2), expected_pixel_2)

## 2. Write the grayscale loop from scratch

Allocate one output value per spatial pixel. Use a single flat loop, then reshape to (h, w). Do not use broadcasting, .mean, or matrix multiplication.

In [36]:
def rgb2grey_loop(x):
    # YOUR CODE
    c, h, w = x.shape
    n = h*w
    print(x.shape)
    x = x.flatten()
    print(x.shape)
    res = torch.empty(n)
    for i in range(n): res[i] = 0.2989 * x[i] + 0.5870 * x[i+n] + 0.1140 * x[i+2*n]
    return res.view(h, w)

expected = 0.2989*tiny[0] + 0.5870*tiny[1] + 0.1140*tiny[2]
assert torch.allclose(rgb2grey_loop(tiny), expected)
assert rgb2grey_loop(tiny).shape == (2, 2)

torch.Size([3, 2, 2])
torch.Size([12])
torch.Size([3, 2, 2])
torch.Size([12])


## 3. Kernel logic versus launch logic

This is intentionally a serial Python simulator. Write a kernel that handles one logical index, then a launcher that invokes it n times. This separation prepares you for CUDA; it is not CPU parallelism.

In [37]:
def grey_kernel(i, flat_x, out, n):
    # YOUR CODE: write exactly out[i]
    out[i] = 0.2989 * flat_x[i] + 0.5870 * flat_x[i+n] + 0.1140 * flat_x[i+2*n]
    pass

def run_kernel(kernel, n, *args):
    # YOUR CODE: serial simulator of n logical threads
    for i in range(n): kernel(i, *args)
    pass

def rgb2grey_kernel_style(x):
    _, h, w = x.shape
    n = h * w
    out = torch.empty(n, dtype=x.dtype, device=x.device)
    run_kernel(grey_kernel, n, x.flatten(), out, n)
    return out.view(h, w)

assert torch.allclose(rgb2grey_kernel_style(tiny), expected)

## 4. Blocks, threads, and bounds

Fill the launch math. For n=1000 and threads=256, exactly 4 blocks launch. The final block covers indices 768–1023, but only 768–999 are valid.

In [38]:
def ceil_div(a, b):
    return int(math.ceil(a/b))

def global_index(block_idx, thread_idx, block_dim):
    return block_idx * block_dim + thread_idx

assert ceil_div(1000, 256) == 4
assert global_index(3, 0, 256) == 768
assert global_index(3, 255, 256) == 1023

def grey_block_kernel(block_idx, thread_idx, block_dim, flat_x, out, n):
    i = global_index(block_idx, thread_idx, block_dim)
    # YOUR CODE: guard invalid i, then write out[i]
    if (i<n): out[i] = 0.2989 * flat_x[i] + 0.5870 * flat_x[i+n] + 0.1140 * flat_x[i+2*n]
    pass

### Explain in your own words

Why does ceiling division deliberately create excess logical threads, and why is the i < n guard required for correctness?

threads are cheap in gpu, if we have 1:1 mapping of threads to data elements, we need at least that many threads. therefore, when we run the kernel we need to guard against the executions of threads for which there is no data elements

## 5. Matrix multiplication — construct the loops

For A[h, k] @ B[k, w], output is C[h, w]. Each C[r, c] is the dot product of row r in A and column c in B. Write the three nested loops. Avoid @, matmul, .mm, einsum, broadcasting, and .sum.

In [41]:
def matmul_loops(a, b):
    ar, ac = a.shape
    br, bc = b.shape
    print((ar, ac), (br, bc))
    t_out = torch.zeros(ar, bc)
    print(t_out)
    for i in range(ar): # 2
        for j in range(bc): # 4
            for k in range(ac): # 3
                t_out[i, j] += a[i, k] * b[k,j]

    return t_out
        

a = torch.tensor([[1., 2., 3.], [4., 5., 6.]])  # (h=2, k=3)
b = torch.tensor([[1., 2., 3., 4.],
                  [5., 6., 7., 8.],
                  [9., 10., 11., 12.]])          # (k=3, w=4)
assert torch.equal(matmul_loops(a, b), a @ b)
assert matmul_loops(a, b).shape == (2, 4)

(2, 3) (3, 4)
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.]])
(2, 3) (3, 4)
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.]])


## 6. Broadcasting — build it one transformation at a time

For fixed row i, a[i, :] has shape (k,). Add a singleton axis to make it (k, 1); multiplying by b[k, w] broadcasts it to (k, w). Summing dimension 0 gives (w,). Fill each expression, then inspect shapes.

In [55]:
i = 0
row = a[i, :]                 # shape: (k,)
# print(a)
print(row)
column = a[i, :, None]            # shape: (k, 1)
print(column)
products = column * b               # shape: (k, w)
out_row = products.sum(dim=0)  # shape: (w,)

assert row.shape == (3,)
assert column.shape == (3, 1)
assert products.shape == (3, 4)
assert out_row.shape == (4,)
assert torch.equal(out_row, (a @ b)[i])

tensor([1., 2., 3.])
tensor([[1.],
        [2.],
        [3.]])


### 6b. Turn one broadcast row into full matmul

Use exactly one Python loop over output rows. Inside it, use the previous broadcasting expression to create each complete output row. Explain why this is generally faster on CPU than three Python loops, despite equivalent math.

In [60]:
def matmul_broadcast_rows(a, b):
    # YOUR CODE: one Python loop; vectorized work inside the loop
    ar, ac = a.shape
    br, bc = b.shape
    t_out = torch.zeros(ar, bc)
    for i in range(ar):
        column = a[i, :, None]
        print(f"\nOutput row {i}: A row {i} reshaped into a column, shape {tuple(column.shape)}. Each value will scale the matching row of B:\n{column}")
        products = column * b
        print(f"Row {i}: element-by-element products after broadcasting against B, shape {tuple(products.shape)}. Each column holds the contributions to one output value:\n{products}")
        out_row = products.sum(dim=0)
        print(f"Row {i}: sum down the rows (dim=0), leaving one total per column, shape {tuple(out_row.shape)}:\n{out_row}")
        t_out[i, :] = out_row
        print(f"Output after storing row {i}. Rows 0 through {i} are complete; any later rows are still initial zeros:\n{t_out}")

    return t_out

assert torch.equal(matmul_broadcast_rows(a, b), a @ b)


Output row 0: A row 0 reshaped into a column, shape (3, 1). Each value will scale the matching row of B:
tensor([[1.],
        [2.],
        [3.]])
Row 0: element-by-element products after broadcasting against B, shape (3, 4). Each column holds the contributions to one output value:
tensor([[ 1.,  2.,  3.,  4.],
        [10., 12., 14., 16.],
        [27., 30., 33., 36.]])
Row 0: sum down the rows (dim=0), leaving one total per column, shape (4,):
tensor([38., 44., 50., 56.])
Output after storing row 0. Rows 0 through 0 are complete; any later rows are still initial zeros:
tensor([[38., 44., 50., 56.],
        [ 0.,  0.,  0.,  0.]])

Output row 1: A row 1 reshaped into a column, shape (3, 1). Each value will scale the matching row of B:
tensor([[4.],
        [5.],
        [6.]])
Row 1: element-by-element products after broadcasting against B, shape (3, 4). Each column holds the contributions to one output value:
tensor([[ 4.,  8., 12., 16.],
        [25., 30., 35., 40.],
        [54., 

_Why is this faster than the three-loop version? Your explanation here._

python does not direct every individual mul and add. it asks pytorch to do whole tensor operations in compiled code, which uses SIMD etc

## 7. 2-D CUDA indexing — translate the CPU loops

Assume tpb.x = tpb.y = 16. Fill global output row/column mapping and flat row-major offsets. Then state which output tile block (x=2, y=1) owns.

In [ ]:
def matmul_thread_indices(block_x, block_y, thread_x, thread_y, block_dim_x=16, block_dim_y=16):
    r = ___
    c = ___
    return r, c

def flat_matmul_offsets(r, c, i, k, w):
    a_offset = ___       # A[r, i]
    b_offset = ___       # B[i, c]
    out_offset = ___     # out[r, c]
    return a_offset, b_offset, out_offset

# assert matmul_thread_indices(2, 1, 0, 0) == (16, 32)
# assert matmul_thread_indices(2, 1, 15, 15) == (31, 47)
# assert flat_matmul_offsets(2, 3, 4, k=5, w=7) == (14, 31, 17)

## 8. Mastery challenge — design the CUDA wrapper and kernel

Without copying from the lecture, write pseudocode or valid C++/CUDA for a float32 matmul extension. Include:

1. CUDA and contiguous input checks.
2. Dimension extraction and size validation.
3. Output allocation preserving input options.
4. A 2-D 16×16 launch grid with ceiling division.
5. Kernel row/column indices, bounds guard, dot-product loop, and output store.
6. A post-launch CUDA error check.

Then explain why this correct naive kernel will usually lose to a @ b: identify at least two reasons.

In [ ]:
# Write your pseudocode or CUDA/C++ answer here.
# Do not run this cell as Python.

## Exercise ledger

| Item | Skill targets | Status | Attempts | Hints | Score | Gap tags | Next drill |
|---|---|---|---:|---:|---:|---|---|
| E1 | tensor-layout, row-major-indexing | Pending | 0 | 0 | — | — | — |
| E2 | loop-construction, tensor-layout | Pending | 0 | 0 | — | — | — |
| E3 | loop-construction, grid-block-thread | Pending | 0 | 0 | — | — | — |
| E4 | bounds-check, grid-block-thread | Pending | 0 | 0 | — | — | — |
| E5 | loop-construction, matmul-work-count | Pending | 0 | 0 | — | — | — |
| E6 | broadcasting-shapes | In progress | 0 | 1 | — | broadcasting-shapes | Revisit shapes with a small concrete row and matrix. |
| E7 | broadcasting-shapes, loop-construction | Pending | 0 | 0 | — | — | — |
| E8 | grid-block-thread, row-major-indexing | Pending | 0 | 0 | — | — | — |
| E9 | cuda-host-device, shared-memory-tiling | Pending | 0 | 0 | — | — | — |

### E6 attempt history

1. Learner requested a worked shape explanation before submitting a solution. Hints: 1. Status: In progress. Score: pending. Gap: broadcasting-shapes.